# Post on historic

timescale db

In [ ]:
# posting historicflow events to Ditto

import json
import pandas as pd
import requests
import os
import asyncio

from flask import json
import httpx
from APItoken import AuthenticationService
import pandas as pd

auth_service = AuthenticationService()

HISTORIC_URL = os.getenv("VCI_HISTORIC_DITTO_URL")
DITTO_URL = os.getenv("VCI_DITTO_URL")
MODELS_DIR = os.getenv("MODELS_DIR", "/app/models")
SERIES_START = pd.Timestamp(os.getenv("SERIES_START", "2022-01-01 00:00:00"))
    
""" Post flow events to the Ditto API. This is used to create the historic flow data in the Ditto API, for model evaluation. """

sublanco_sentido_to_sensor = {
    ("Campo Alegre - Boavista",  1.0): 741,
    ("Campo Alegre - Boavista", -1.0): 741,

    ("Paranhos - Amial",  1.0): 722,
    ("Paranhos - Amial", -1.0): 722,

    ("A3 - Paranhos",  1.0): 3,
    ("A3 - Paranhos", -1.0): 3,

    ("Antas - A3",  1.0): 716,
    ("Antas - A3", -1.0): 716,

    ("Mercado Abastecedor - Antas",  1.0): 719,
    ("Mercado Abastecedor - Antas", -1.0): 719,

    ("Freixo(A43) - Mercado Abastecedor",  1.0): 718,
    ("Freixo(A43) - Mercado Abastecedor", -1.0): 718,
}

def create_events(
    timestamps,
    flows,
    sublanco,
    sentido,
):

    events = []

    for (ts, flow) in zip(timestamps, flows):

        ts_iso = pd.Timestamp(ts).isoformat()

        detector_id = sublanco_sentido_to_sensor.get((sublanco, sentido))
        if detector_id is None:
            raise ValueError(f"No sensor mapped for {sublanco!r}, {sentido!r}")

        print(f"historicFlowData:{detector_id}_{float(sentido)}")
        event = {
            "time": ts_iso,
            "thing_id": f"historicFlowData:{detector_id}_{float(sentido)}",
            "action": "modified",
            "revision": 1,
            "path": "/features/traffic/",
            "value": {"properties": {
                "timestamp": ts_iso,
                "flows": float(flow),   
                "sentido": float(sentido),
            }}
        }

        events.append(event)
    
    return events



In [19]:
import pandas as pd
import os

for file_name in os.listdir("data/"):
    if file_name.endswith(".csv"):
        df = pd.read_csv(os.path.join("data/", file_name), sep=';')

        sublanco = df["sublanco"].iloc[0] if "sublanco" in df.columns else None
        sentido = df["sentido"].iloc[0] if "sentido" in df.columns else None
        
        print(f"{sublanco}; {sentido}")

        

Antas - A3; 1.0
Campo Alegre - Boavista; 1.0
Mercado Abastecedor - Antas; 1.0
Mercado Abastecedor - Antas; -1.0
Freixo(A43) - Mercado Abastecedor; -1.0
Paranhos - Amial; -1.0
Antas - A3; -1.0
Freixo(A43) - Mercado Abastecedor; 1.0
A3 - Paranhos; -1.0
Campo Alegre - Boavista; -1.0
Paranhos - Amial; 1.0
A3 - Paranhos; 1.0


In [20]:
import os
import asyncio
import httpx

from APItoken import AuthenticationService

auth_service = AuthenticationService()

HISTORIC_URL = os.getenv("VCI_HISTORIC_DITTO_URL")

print(HISTORIC_URL)
BATCH_SIZE = 2000


async def post_events(events):

    url = f"{HISTORIC_URL}events"

    bearer_token = auth_service.get_token()

    headers = {
        "Authorization": f"Bearer {bearer_token}",
                }
    

    async with httpx.AsyncClient(timeout=30) as client:

        for i in range(0, len(events), BATCH_SIZE):

            batch = {
                "events": events[i:i + BATCH_SIZE]
            }

            resp = await client.post(
                url,
                headers=headers,
                json=batch
            )

            resp.raise_for_status()

            print(
                f"Batch {i // BATCH_SIZE + 1} OK "
                f"({resp.status_code})"
            )


https://dt4mob.av.it.pt/historic/


In [21]:
from scripts.createTrainflows import create_train_set

for file_name in os.listdir("data/"):  
    if file_name.endswith(".csv"):  
        print(f"Processing {file_name}...")
 
        y_train, train_timestamps, sublanco, sentido = create_train_set(data_dir="data", dataset_name=file_name)

        
        events= create_events(train_timestamps, y_train, sublanco, sentido)
        print(events[0])    
        print(events[-1])
        result = await post_events(events)    

Processing Antas_-_A3_1.0.csv...
{'time': '2022-01-01T00:00:00', 'thing_id': 'historicFlowData:716_1.0', 'action': 'modified', 'revision': 1, 'path': '/features/traffic/', 'value': {'properties': {'timestamp': '2022-01-01T00:00:00', 'flows': 1151.0, 'sentido': 1.0}}}
{'time': '2024-07-14T14:00:00', 'thing_id': 'historicFlowData:716_1.0', 'action': 'modified', 'revision': 1, 'path': '/features/traffic/', 'value': {'properties': {'timestamp': '2024-07-14T14:00:00', 'flows': 3437.0, 'sentido': 1.0}}}
Batch 1 OK (201)
Batch 2 OK (201)
Batch 3 OK (201)
Batch 4 OK (201)
Batch 5 OK (201)
Batch 6 OK (201)
Batch 7 OK (201)
Batch 8 OK (201)
Batch 9 OK (201)
Batch 10 OK (201)
Batch 11 OK (201)
Batch 12 OK (201)
Processing Campo_Alegre_-_Boavista_1.0.csv...
{'time': '2022-01-01T00:00:00', 'thing_id': 'historicFlowData:741_1.0', 'action': 'modified', 'revision': 1, 'path': '/features/traffic/', 'value': {'properties': {'timestamp': '2022-01-01T00:00:00', 'flows': 1093.0, 'sentido': 1.0}}}
{'time': 

In [22]:
from scripts.createTestFlows import create_test_set

for file_name in os.listdir("data/"):  
    if file_name.endswith(".csv"):  
        print(f"Processing {file_name}...")
        # # skip files that already in api (oops)
        # if file_name not in ["Antas_-_A3_1.0.csv", "Campo_Alegre_-_Boavista_1.0.csv", "Mercado_Abastecedor_-_Antas_1.0.csv", "Mercado_Abastecedor_-_Antas_-1.0.csv", "Freixo(A43)_-_Mercado_Abastecedor_-1.0.csv", "Paranhos_-_Amial_-1.0.csv"]:
            
        y_test, test_timestamps, sublanco, sentido = create_test_set(data_dir="data", dataset_name=file_name)

        events= create_events(test_timestamps, y_test, sublanco, sentido)
        print(events[0])    
        print(events[-1])
        result = await post_events(events)    

Processing Antas_-_A3_1.0.csv...
{'time': '2024-07-14T15:00:00', 'thing_id': 'historicFlowData:716_1.0', 'action': 'modified', 'revision': 1, 'path': '/features/traffic/', 'value': {'properties': {'timestamp': '2024-07-14T15:00:00', 'flows': 4454.0, 'sentido': 1.0}}}
{'time': '2024-12-31T23:00:00', 'thing_id': 'historicFlowData:716_1.0', 'action': 'modified', 'revision': 1, 'path': '/features/traffic/', 'value': {'properties': {'timestamp': '2024-12-31T23:00:00', 'flows': 1010.0, 'sentido': 1.0}}}
Batch 1 OK (201)
Batch 2 OK (201)
Batch 3 OK (201)
Processing Campo_Alegre_-_Boavista_1.0.csv...
{'time': '2024-07-14T15:00:00', 'thing_id': 'historicFlowData:741_1.0', 'action': 'modified', 'revision': 1, 'path': '/features/traffic/', 'value': {'properties': {'timestamp': '2024-07-14T15:00:00', 'flows': 3135.0, 'sentido': 1.0}}}
{'time': '2024-12-31T23:00:00', 'thing_id': 'historicFlowData:741_1.0', 'action': 'modified', 'revision': 1, 'path': '/features/traffic/', 'value': {'properties': {'

In [37]:
from scripts.createTestFlows import create_test_set

for file_name in os.listdir("data/"):  
    if file_name.endswith(".csv"):  
        print(f"Processing {file_name}...")
        # # skip files that already in api (oops)
        # if file_name not in ["Antas_-_A3_1.0.csv", "Campo_Alegre_-_Boavista_1.0.csv", "Mercado_Abastecedor_-_Antas_1.0.csv", "Mercado_Abastecedor_-_Antas_-1.0.csv", "Freixo(A43)_-_Mercado_Abastecedor_-1.0.csv", "Paranhos_-_Amial_-1.0.csv"]:
            
        y_test, test_timestamps, sublanco, sentido = create_test_set(data_dir="data", dataset_name=file_name)

        print(len(y_test))
        break

Processing Antas_-_A3_1.0.csv...
4089


# Post football game info and meteorological info to historic API

In [67]:
# from "2024-07-14T15:00:00" to "2024-12-31T23:00:00" 

# posting historicflow events to Ditto

import json
import pandas as pd
import requests
import os
import asyncio

from flask import json
import httpx
from APItoken import AuthenticationService
import pandas as pd

auth_service = AuthenticationService()

HISTORIC_URL = os.getenv("VCI_HISTORIC_DITTO_URL")
    
""" Post meteorological and football events to the Ditto API. This is used to create the historic flow data in the Ditto API, for model evaluation. """

def create_meteorological_events(
    timestamps,
    temperature,
    precipitation
):
    """ Create a list with weather data with daily frequency"""

    events = []

    for (ts, temperature, precipitation) in zip(timestamps, temperature, precipitation):

        ts_iso = pd.Timestamp(ts).isoformat()

        event = {
            "time": ts_iso,
            "thing_id": f"historicWeatherData:dailyAverage",
            "action": "modified",
            "revision": 1,
            "path": "/features/historicweather/",
            "value": {"properties": {
                "timestamp": ts_iso,
                "temperature": round(float(temperature),2),   
                "precipitation": round(float(precipitation),2),
            }}
        }

        events.append(event)
    
    return events

In [68]:
def create_football_events(
    timestamps,
):

    events = []

    for ts in timestamps:

        ts_iso = pd.Timestamp(ts).isoformat()

        event = {
            "time": ts_iso,
            "thing_id": f"footballGames:EstadioDragao",
            "action": "modified",
            "revision": 1,
            "path": "/features/historicfootball/",
            "value": {"properties": {
                "timestamp": ts_iso,
            }}
        }

        events.append(event)
    print(len(events))
    
    return events

In [69]:
import pandas as pd
weather = pd.read_csv("data/weather/weather_data.csv")
# from "2024-07-14T15:00:00" to "2024-12-31T23:00:00"

#combine columns year	month	day into one column timestamp
weather["timestamp"] = pd.to_datetime(weather[['year', 'month', 'day']])

# if timestamp # from "2024-07-14T15:00:00" to "2024-12-31T23:00:00"

mask = (weather['timestamp'] >= '2022-01-01') & (weather['timestamp'] <= '2024-12-31')
filtered_weather = weather.loc[mask]

weather_events = create_meteorological_events(
    filtered_weather['timestamp'], 
    filtered_weather['Avg_Temperature'],    
    filtered_weather['Avg_Precipitation']
)

weather_events

[{'time': '2022-01-01T00:00:00',
  'thing_id': 'historicWeatherData:dailyAverage',
  'action': 'modified',
  'revision': 1,
  'path': '/features/historicweather/',
  'value': {'properties': {'timestamp': '2022-01-01T00:00:00',
    'temperature': 17.2,
    'precipitation': 2.93}}},
 {'time': '2022-01-02T00:00:00',
  'thing_id': 'historicWeatherData:dailyAverage',
  'action': 'modified',
  'revision': 1,
  'path': '/features/historicweather/',
  'value': {'properties': {'timestamp': '2022-01-02T00:00:00',
    'temperature': 13.7,
    'precipitation': 1.27}}},
 {'time': '2022-01-03T00:00:00',
  'thing_id': 'historicWeatherData:dailyAverage',
  'action': 'modified',
  'revision': 1,
  'path': '/features/historicweather/',
  'value': {'properties': {'timestamp': '2022-01-03T00:00:00',
    'temperature': 11.95,
    'precipitation': 0.07}}},
 {'time': '2022-01-04T00:00:00',
  'thing_id': 'historicWeatherData:dailyAverage',
  'action': 'modified',
  'revision': 1,
  'path': '/features/historic

In [73]:
len(weather_events)

1096

In [70]:
result = await post_events(weather_events)  
result

Batch 1 OK (201)


In [71]:
import pandas as pd
games = pd.read_csv("data/futebol/games.csv")
# from "2024-07-14T15:00:00" to "2024-12-31T23:00:00"


games["timestamp"] = pd.to_datetime(games['Data'] + ' ' + games['Horário'])
games
# if timestamp # from "2024-07-14T15:00:00" to "2024-12-31T23:00:00"

mask = (games['timestamp'] >= '2022-01-01') & (games['timestamp'] <= '2024-12-31')
filtered_games = games.loc[mask]

games_events = create_football_events(
    filtered_games['timestamp']
)

games_events

78


[{'time': '2022-01-23T20:30:00',
  'thing_id': 'footballGames:EstadioDragao',
  'action': 'modified',
  'revision': 1,
  'path': '/features/historicfootball/',
  'value': {'properties': {'timestamp': '2022-01-23T20:30:00'}}},
 {'time': '2022-01-30T20:30:00',
  'thing_id': 'footballGames:EstadioDragao',
  'action': 'modified',
  'revision': 1,
  'path': '/features/historicfootball/',
  'value': {'properties': {'timestamp': '2022-01-30T20:30:00'}}},
 {'time': '2022-02-11T20:15:00',
  'thing_id': 'footballGames:EstadioDragao',
  'action': 'modified',
  'revision': 1,
  'path': '/features/historicfootball/',
  'value': {'properties': {'timestamp': '2022-02-11T20:15:00'}}},
 {'time': '2022-02-27T20:30:00',
  'thing_id': 'footballGames:EstadioDragao',
  'action': 'modified',
  'revision': 1,
  'path': '/features/historicfootball/',
  'value': {'properties': {'timestamp': '2022-02-27T20:30:00'}}},
 {'time': '2022-03-13T18:00:00',
  'thing_id': 'footballGames:EstadioDragao',
  'action': 'modif

In [72]:
result = await post_events(games_events)  
result

Batch 1 OK (201)


# Delete testing

In [3]:
import os
import asyncio
import httpx
import json
import pandas as pd
import requests
import os
import asyncio

from flask import json
import httpx
from APItoken import AuthenticationService
import pandas as pd

auth_service = AuthenticationService()

HISTORIC_URL = os.getenv("VCI_HISTORIC_DITTO_URL")

print(HISTORIC_URL)
BATCH_SIZE = 2000


async def delete_events(detector_id, since, until):

    url = f"{HISTORIC_URL}events/{detector_id}"

    # print(url)
    bearer_token = auth_service.get_token()

    # print(f"Bearer token: {bearer_token}")
    headers = {
        "Authorization": f"Bearer {bearer_token}",
                }
    
    params ={"since": since,
             "until": until}

    async with httpx.AsyncClient(timeout=300) as client:
        resp = await client.delete(
            url,
            headers=headers,
            params=params
        )

        res= resp.raise_for_status()
    return res

            

https://dt4mob.av.it.pt/historic/


In [2]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
# sensors = [ 
#   "historicFlowData:3",
#   "historicFlowData:716",
#   "historicFlowData:718",
#   "historicFlowData:719",
#   "historicFlowData:722",
#   "historicFlowData:741"
# ]
sensors=[
  "historicVCIpredictions:3_-1.0",
  "historicVCIpredictions:3_1.0",
  "historicVCIpredictions:716_-1.0",
  "historicVCIpredictions:716_1.0",
  "historicVCIpredictions:718_-1.0",
  "historicVCIpredictions:718_1.0",
  "historicVCIpredictions:719_-1.0",
  "historicVCIpredictions:719_1.0",
  "historicVCIpredictions:722_-1.0",
  "historicVCIpredictions:722_1.0",
  "historicVCIpredictions:741_-1.0",
  "historicVCIpredictions:741_1.0",
  "historicVCIpredictionsClassic:3_-1.0",
  "historicVCIpredictionsClassic:3_1.0",
  "historicVCIpredictionsClassic:716_-1.0",
  "historicVCIpredictionsClassic:716_1.0",
  "historicVCIpredictionsClassic:718_-1.0",
  "historicVCIpredictionsClassic:718_1.0",
  "historicVCIpredictionsClassic:719_-1.0",
  "historicVCIpredictionsClassic:719_1.0",
  "historicVCIpredictionsClassic:722_-1.0",
  "historicVCIpredictionsClassic:722_1.0",
  "historicVCIpredictionsClassic:741_-1.0",
  "historicVCIpredictionsClassic:741_1.0"
]

start_bound = datetime.fromisoformat("2022-01-01T00:00:00")
end_bound = datetime.fromisoformat("2024-12-31T23:59:59")

async def main():
    for sensor in sensors:
        print(sensor)
        current_start = start_bound


        while current_start < end_bound:
            current_end = current_start + relativedelta(months=1)
            
            if current_end > end_bound:
                current_end = end_bound
                
            since_str = current_start.isoformat()
            until_str = current_end.isoformat()
            
            res = await delete_events(sensor, since_str, until_str)
                
            current_start = current_end

            print(f"Deleted {sensor} from {since_str} to {until_str} -- {res}")
    print("!!!Deleted all")

In [4]:
await main()

historicVCIpredictions:3_-1.0
Deleted historicVCIpredictions:3_-1.0 from 2022-01-01T00:00:00 to 2022-02-01T00:00:00 -- <Response [204 No Content]>
Deleted historicVCIpredictions:3_-1.0 from 2022-02-01T00:00:00 to 2022-03-01T00:00:00 -- <Response [204 No Content]>
Deleted historicVCIpredictions:3_-1.0 from 2022-03-01T00:00:00 to 2022-04-01T00:00:00 -- <Response [204 No Content]>
Deleted historicVCIpredictions:3_-1.0 from 2022-04-01T00:00:00 to 2022-05-01T00:00:00 -- <Response [204 No Content]>
Deleted historicVCIpredictions:3_-1.0 from 2022-05-01T00:00:00 to 2022-06-01T00:00:00 -- <Response [204 No Content]>
Deleted historicVCIpredictions:3_-1.0 from 2022-06-01T00:00:00 to 2022-07-01T00:00:00 -- <Response [204 No Content]>
Deleted historicVCIpredictions:3_-1.0 from 2022-07-01T00:00:00 to 2022-08-01T00:00:00 -- <Response [204 No Content]>
Deleted historicVCIpredictions:3_-1.0 from 2022-08-01T00:00:00 to 2022-09-01T00:00:00 -- <Response [204 No Content]>
Deleted historicVCIpredictions:3_-